<span style="color: #6a737d; font-family: monospace;">
Created on Sat Feb 08 2025 13:03:45<br>
Author: Mukai (Tom Notch) Yu<br>
Email: mukaiy@andrew.cmu.edu<br>
Affiliation: Carnegie Mellon University, Robotics Institute<br>
<br>
Copyright Ⓒ 2025 Mukai (Tom Notch) Yu<br>
</span>

In [ ]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# %cd $USF_PROJECT_DIRECTORY doesn't work here because it's set by os.environ, not before the notebook starts
%cd ../..
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
import torch.optim as optim
from hydra import compose, initialize
from hydra.utils import instantiate
from pytorch_lightning import LightningDataModule, Trainer
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm

from usf.dataset.mnist import MNISTDataset
from usf.network.model.mnist import MNISTLightningModel, PlanarMnist, SphericalMnist
from usf.visualization.spherical_projection import visualize_spherical_image

In [ ]:
batch_size = 16

## Read Config

In [ ]:
with initialize(
    version_base=None,
    config_path="../../config",
):  # hydra doesn't respect the current working directory
    config = compose(
        config_name="default.yaml",
        overrides=[
            "task=mnist",
            "task.data_module.num_workers=1",
            f"task.data_module.batch_size={batch_size}",
            "task.trainer.log_every_n_steps=1",
            "~task.trainer.strategy",  # no strategy
            "~task.trainer.logger",  # disable logging
            "~task.trainer.profiler",  # disable profiler
            "+task.trainer.enable_progress_bar=True",  # print progress in notebook
            "+enable_checkpointing=False",  # disable checkpointing
        ],
    )

## Single Batch Overfit Test

In [ ]:
datamodule: LightningDataModule = instantiate(config.task.data_module)
datamodule.setup()  # must setup() because that instantiates self.train_dataset and self.val_dataset
single_batch_datamodule = batch_size @ datamodule
single_batch_datamodule.val_dataloader = (
    lambda: []
)  # override val_dataloader function to provide empty validation dataloader to skip validation

In [ ]:
with initialize(version_base=None, config_path="../../config"):
    planar_config = compose(
        config_name="default.yaml",
        overrides=[
            "task=mnist",
            "task/mnist@task.architecture=planar",
        ],
    )
    planar_model: MNISTLightningModel = instantiate(planar_config.task.model)

In [ ]:
with initialize(version_base=None, config_path="../../config"):
    spherical_config = compose(
        config_name="default.yaml",
        overrides=[
            "task=mnist",
            "task/mnist@task.architecture=spherical",
        ],
    )
    spherical_model: MNISTLightningModel = instantiate(spherical_config.task.model)

In [ ]:
trainer: Trainer = instantiate(config.task.trainer)

In [ ]:
trainer.fit(planar_model, single_batch_datamodule)

In [ ]:
# re-instantiate the trainer to reset the state
trainer: Trainer = instantiate(config.task.trainer)

In [ ]:
trainer.fit(spherical_model, single_batch_datamodule)

In [ ]:
MNIST_dataset_base_path = "data/MNIST"
batch_size = 1024
augmentation = {"rotation": 1.0}
seed = "MNIST Classification"
epochs = 10
device = "cuda" if torch.cuda.is_available() else "cpu"

# Load MNIST

In [ ]:
train_dataset = MNISTDataset(
    dataset_base_path=MNIST_dataset_base_path,
    dataset_type="train",
    augmentation=augmentation,
)

In [ ]:
test_dataset = MNISTDataset(
    dataset_base_path=MNIST_dataset_base_path,
    dataset_type="test",
    augmentation=augmentation,
    seed=seed,
)

In [ ]:
train_dataloader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=train_dataset.collate_fn,
)

In [ ]:
test_dataloader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=test_dataset.collate_fn,
)

In [ ]:
# Retrieve one batch from the DataLoader
batch = next(iter(train_dataloader))

Skip the following if batch size is huge

In [ ]:
# Visualize the batch using the internal function from dataset class
fig = train_dataset.visualize_batch(batch)
plt.show()

In [ ]:
plt.imshow(
    visualize_spherical_image(
        batch["inputs"]["spherical_images"],
        point_size=10,
        mode="offscreen",
    )
)
plt.axis("off")
plt.show()

# MNIST Classification Network

In [ ]:
def train(model, dataloader, optimizer, epoch):
    model.train()
    running_loss = 0.0
    # Wrap the dataloader with tqdm for a progress bar
    pbar = tqdm(dataloader, desc=f"Epoch {epoch} Training", leave=False)
    for batch in pbar:
        batch = dataloader.dataset.move_batch_to(
            batch, device=device, dtype=torch.float32
        )
        optimizer.zero_grad()
        if isinstance(model, PlanarMnist):
            # Convert from (batch_size, height, width, channels) to (batch_size, channels, height, width)
            images = batch["inputs"]["images"].permute(0, 3, 1, 2)
            outputs = model(images)
        elif isinstance(model, SphericalMnist):
            spherical_images = batch["inputs"]["spherical_images"]
            outputs = model(spherical_images)
        labels = batch["labels"].long()
        loss = F.cross_entropy(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        # Update tqdm progress bar with the current loss
        pbar.set_postfix(loss=f"{loss.item():.4f}")
    avg_loss = running_loss / len(dataloader)
    return avg_loss

In [ ]:
def test(model, dataloader):
    model.eval()
    test_loss = 0.0
    correct = 0
    total = 0
    pbar = tqdm(dataloader, desc="Testing", leave=False)
    with torch.no_grad():
        for batch in pbar:
            batch = dataloader.dataset.move_batch_to(
                batch, device=device, dtype=torch.float32
            )
            if isinstance(model, PlanarMnist):
                images = batch["inputs"]["images"].permute(0, 3, 1, 2)
                outputs = model(images)
            elif isinstance(model, SphericalMnist):
                spherical_images = batch["inputs"]["spherical_images"]
                outputs = model(spherical_images)
            labels = batch["labels"].long()
            loss = F.cross_entropy(outputs, labels, reduction="sum")
            test_loss += loss.item()
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
            pbar.set_postfix(loss=f"{loss.item():.4f}")
    avg_test_loss = test_loss / total
    accuracy = 100.0 * correct / total
    print(
        f"Test Loss: {avg_test_loss:.4f}, Accuracy: {correct}/{total} ({accuracy:.2f}%)"
    )
    return avg_test_loss

In [ ]:
weighting_function_config = {
    "distance": {
        # "function": "continuous",
        # "activation": "ReLU",
        # "hidden_dims": [8, 8],
        # "embedding": {"type": "cosine", "L": 3},
        "function": "discrete",
        "num_slices": 3,
    }
}
spherical_mnist_model = SphericalMnist(
    location_sampler="icosahedron", weighting_function_config=weighting_function_config
).to(device)
spherical_optimizer = optim.Adam(spherical_mnist_model.parameters(), lr=0.001)

In [ ]:
planar_mnist_model = PlanarMnist().to(device)
planar_optimizer = optim.Adam(planar_mnist_model.parameters(), lr=0.001)

In [ ]:
spherical_train_losses = []
spherical_eval_losses = []
planar_train_losses = []
planar_eval_losses = []

In [ ]:
tensor = torch.randn(1, 3, 4)
print(tensor)

In [ ]:
torch.amax(tensor, dim=1, keepdim=True).expand(-1, 3, -1)[:, 0, :].shape

In [ ]:
print("==== Training Spherical CNN ====")
for epoch in range(1, epochs + 1):
    spherical_train_loss = train(
        spherical_mnist_model, train_dataloader, spherical_optimizer, epoch
    )
    spherical_eval_loss = test(spherical_mnist_model, test_dataloader)
    spherical_train_losses.append(spherical_train_loss)
    spherical_eval_losses.append(spherical_eval_loss)

In [ ]:
print("==== Training Planar CNN ====")
for epoch in range(1, epochs + 1):
    planar_train_loss = train(
        planar_mnist_model, train_dataloader, planar_optimizer, epoch
    )
    planar_eval_loss = test(planar_mnist_model, test_dataloader)
    planar_train_losses.append(planar_train_loss)
    planar_eval_losses.append(planar_eval_loss)

In [ ]:
epoch_range = range(1, epochs + 1)
plt.figure(figsize=(10, 6))
plt.plot(epoch_range, planar_train_losses, marker="o", label="Planar Train Loss")
plt.plot(epoch_range, planar_eval_losses, marker="o", label="Planar Eval Loss")
plt.plot(epoch_range, spherical_train_losses, marker="x", label="Spherical Train Loss")
plt.plot(epoch_range, spherical_eval_losses, marker="x", label="Spherical Eval Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Evaluation Loss vs Epochs")
plt.legend()
plt.grid(True)
plt.show()

# Visualize Layers

In [ ]:
from usf.visualization.spherical_layer import visualize_spherical_channels

get the first image

In [ ]:
sample_batch_spherical_images = batch["inputs"]["spherical_images"][:1]

In [ ]:
visualize_spherical_image(sample_batch_spherical_images, point_size=10)

In [ ]:
x = sample_batch_spherical_images
batch_size = len(x)

### First Layer Group

In [ ]:
x = spherical_mnist_model.conv_block["conv1"](x)
x = spherical_mnist_model.conv_block["act1"](x)
first_layer_output = spherical_mnist_model.conv_block["pool1"](x)

In [ ]:
visualize_spherical_image(
    visualize_spherical_channels(first_layer_output)[0], point_size=10, fps=1
)

### Second Layer Group

In [ ]:
x = spherical_mnist_model.conv_block["conv2"](x)
x = spherical_mnist_model.conv_block["act2"](x)
second_layer_output = spherical_mnist_model.conv_block["pool2"](x)

In [ ]:
visualize_spherical_image(
    visualize_spherical_channels(second_layer_output)[0], point_size=10, fps=1
)

### Fully Connected Layers

In [ ]:
batch_value = second_layer_output.batch_value
batch_value = batch_value.view(batch_size, -1)
batch_value = F.relu(spherical_mnist_model.fc1(batch_value))
batch_value = spherical_mnist_model.fc2(batch_value)

In [ ]:
batch_value